# 09 章 · Capstone:Mini-R1 复现(GSM8K-zh)

## 这是什么
把第 5-6 章串起来,在 Qwen2.5-1.5B 上复现 **DeepSeek-R1-Zero / nanochat / Open-R1** 的核心路线:
**SFT(可选)→ GRPO 用 verifiable reward 在 GSM8K-zh 上训 → accuracy 真实提升**。

跑通这一节,你就能在面试里 5 分钟讲一个完整、有数字、有踩坑的项目故事 ——
**比讲 10 个 LoRA 100 行代码有用得多**。

## 目标数字(可达)
| 阶段 | GSM8K-zh test accuracy |
|---|---|
| Qwen2.5-1.5B-Instruct baseline(zero-shot) | ~8-12% |
| + SFT on Alpaca-zh + GSM8K-zh train | ~12-18% |
| + GRPO with verifiable reward(200 步) | ~22-30% |

具体数字与超参、训练步数、采样温度都有关。**有真实差异**就是胜利。

## ⚠️ 运行要求
- **Linux + CUDA GPU 12-16GB**(GRPO 显存吃 `num_generations`)
- 时间预算:**baseline 10 min + GRPO 200 步 2-4h**(单卡 4070)
- 若没卡:把 `N_TEST_EVAL = 30`,`MAX_STEPS = 50`,本地走一遍流程

## 对应八股
`liguodongiot/llm-action`:`llm-interview/llm-rlhf.md` + `llm-alignment/`


## 1. 环境检查


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from scripts.env_check import check
check()


## 2. 加载模型 + 测试集

起点用 `Qwen2.5-1.5B-Instruct` —— 它已经做过 instruction tuning,
不再需要从 base 起 SFT(那是第 05 章干的事)。本节专注 GRPO 的提升。

> 进阶玩法:把 `MODEL_NAME` 换成你在 `05_sft/03_unsloth_lora.ipynb` 训出来的 LoRA-merged 模型,
> 数字会更亮。


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# 或:MODEL_NAME = "../checkpoints/05_sft_unsloth_lora/merged_16bit"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, device_map=device)

train_set = load_dataset("meta-math/GSM8K_zh", split="train")
test_set  = load_dataset("meta-math/GSM8K_zh", split="test")
print(f"train: {len(train_set)}  |  test: {len(test_set)}")


## 3. 通用 reward / 评测函数(从 06_alignment/03 抽出)


In [ ]:
import re

THINK_RE  = re.compile(r"<think>(.*?)</think>", re.DOTALL)
ANSWER_RE = re.compile(r"<answer>(.*?)</answer>", re.DOTALL)
NUMBER_RE = re.compile(r"-?\d+(?:\.\d+)?")

SYSTEM_PROMPT = """你是一个善于推理的数学助手。请严格按照以下格式回答:
<think>
在这里写出详细的推理步骤
</think>
<answer>
在这里写出最终的数字答案,只写一个数字
</answer>"""


def make_prompt(q_zh: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": q_zh},
    ]


def extract_answer(text: str) -> str | None:
    m = ANSWER_RE.search(text)
    if not m:
        # fallback:取文本里最后一个数字
        nums = NUMBER_RE.findall(text)
        return nums[-1] if nums else None
    nums = NUMBER_RE.findall(m.group(1))
    return nums[-1] if nums else None


def check_correct(pred: str | None, gt: str) -> bool:
    if pred is None:
        return False
    try:
        return abs(float(pred) - float(gt)) < 1e-3
    except ValueError:
        return False


# ---- reward 函数(GRPO 用) ----
def format_reward(completions, **kwargs):
    return [
        0.5 if (THINK_RE.search(c[0]["content"]) and ANSWER_RE.search(c[0]["content"])) else 0.0
        for c in completions
    ]

def answer_reward(completions, ground_truth=None, **kwargs):
    out = []
    for completion, gt in zip(completions, ground_truth):
        pred = extract_answer(completion[0]["content"])
        out.append(1.5 if check_correct(pred, gt) else 0.0)
    return out


## 4. Baseline 评测(训练前)


In [ ]:
from tqdm import tqdm

N_TEST_EVAL = 100   # 100 题足够看趋势,1k 题才稳;教学跑用 100

def eval_accuracy(model, tokenizer, test_set, n: int, label: str = "eval") -> dict:
    model.eval()
    correct, total = 0, 0
    has_think, has_answer = 0, 0
    samples = test_set.shuffle(seed=0).select(range(n))

    with torch.no_grad():
        for ex in tqdm(samples, desc=label):
            prompt = make_prompt(ex["question_zh"])
            text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(text, return_tensors="pt").to(model.device)
            out = model.generate(
                **inputs,
                max_new_tokens = 400,
                do_sample = False,                # 评测用 greedy
                pad_token_id = tokenizer.eos_token_id,
            )
            response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
            pred = extract_answer(response)
            gt = str(ex["answer_only"]).strip()
            correct += int(check_correct(pred, gt))
            total += 1
            has_think  += bool(THINK_RE.search(response))
            has_answer += bool(ANSWER_RE.search(response))

    return {
        "n": total,
        "accuracy": correct / total,
        "format_think_pct": has_think / total,
        "format_answer_pct": has_answer / total,
    }


print("=== 训练前 baseline 评测 ===")
baseline = eval_accuracy(model, tokenizer, test_set, N_TEST_EVAL, "baseline")
print(baseline)


## 5. 准备 GRPO 训练数据(从 train split,不要碰 test)


In [ ]:
SUBSET_TRAIN = 500  # 教学跑 500;真复现用全部 7.5k

def prepare(example):
    return {
        "prompt": make_prompt(example["question_zh"]),
        "ground_truth": str(example["answer_only"]).strip(),
    }

grpo_train = (train_set
              .shuffle(seed=42)
              .select(range(SUBSET_TRAIN))
              .map(prepare, remove_columns=train_set.column_names))
print(f"GRPO 训练样本: {len(grpo_train)}")


## 6. GRPO 训练 200 步


In [ ]:
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = "../checkpoints/09_capstone_mini_r1"
MAX_STEPS = 200

cfg = GRPOConfig(
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_generations = 4,
    max_completion_length = 320,
    max_prompt_length = 512,

    learning_rate = 5e-6,
    lr_scheduler_type = "cosine",
    warmup_steps = 10,
    max_steps = MAX_STEPS,
    optim = "adamw_torch",
    weight_decay = 0.01,
    beta = 0.04,

    temperature = 0.9,
    top_p = 0.95,

    bf16 = True,
    gradient_checkpointing = True,
    report_to = "none",
    logging_steps = 5,
    save_steps = 100,
    save_total_limit = 2,
    seed = 42,
)

trainer = GRPOTrainer(
    model = model,
    args = cfg,
    train_dataset = grpo_train,
    reward_funcs = [format_reward, answer_reward],
    processing_class = tokenizer,
)

import time
t0 = time.time()
stats = trainer.train()
print(f"训练耗时:{(time.time()-t0)/60:.1f} min  |  末步 loss {stats.training_loss:.4f}")


## 7. 训练后评测(对照 baseline)


In [ ]:
print("=== 训练后评测 ===")
after = eval_accuracy(model, tokenizer, test_set, N_TEST_EVAL, "after-grpo")
print(after)

print("\n" + "=" * 50)
print(f"  指标         | baseline | after | 提升")
print("-" * 50)
for k in ["accuracy", "format_think_pct", "format_answer_pct"]:
    b = baseline[k] * 100
    a = after[k] * 100
    print(f"  {k:18s} | {b:6.1f}%  | {a:5.1f}% | +{a-b:+.1f} pt")
print("=" * 50)


## 8. 看一些训练后的样本(`<think>` 涌现了吗?)


In [ ]:
samples = test_set.shuffle(seed=99).select(range(3))
for ex in samples:
    prompt = make_prompt(ex["question_zh"])
    text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=400, do_sample=True,
                          temperature=0.7, top_p=0.9, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"\n>>> 题目:{ex['question_zh']}")
    print(f">>> 标准答案:{ex['answer_only']}")
    print(f">>> 模型回答:\n{response}")
    print("-" * 60)


## 9. 完整曲线:reward + accuracy


In [ ]:
import matplotlib.pyplot as plt

hist = trainer.state.log_history
def get(key):
    return [(l["step"], l[key]) for l in hist if key in l]

fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))

for key, label, color in [
    ("reward", "total", "C0"),
    ("rewards/format_reward", "format", "C1"),
    ("rewards/answer_reward", "answer", "C2"),
]:
    s = get(key)
    if s:
        x, y = zip(*s); ax[0].plot(x, y, label=label, color=color)
ax[0].set_title("GRPO rewards"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[0].set_xlabel("step")

s = get("loss")
if s:
    x, y = zip(*s); ax[1].plot(x, y)
ax[1].set_title("policy loss"); ax[1].grid(alpha=0.3)
ax[1].set_xlabel("step")

# accuracy 对比
ax[2].bar(["baseline", "after GRPO"],
          [baseline["accuracy"]*100, after["accuracy"]*100],
          color=["gray", "C3"])
ax[2].set_ylabel("GSM8K-zh test accuracy (%)")
ax[2].set_title(f"Capstone: {baseline['accuracy']*100:.1f}% → {after['accuracy']*100:.1f}%")
for i, v in enumerate([baseline["accuracy"]*100, after["accuracy"]*100]):
    ax[2].text(i, v + 0.5, f"{v:.1f}%", ha="center")
ax[2].grid(alpha=0.3, axis="y")

plt.tight_layout(); plt.show()


## 10. 面试讲解模板(5 分钟版)

> 直接背下面这段,数字换成你跑出来的。

---

**1. 项目目标(15s)**

在 12GB 消费级 GPU 上,用 Qwen2.5-1.5B-Instruct 复现 DeepSeek-R1-Zero 的核心路线 ——
**用 verifiable reward + GRPO** 在中文数学推理任务(GSM8K-zh)上拿到可衡量提升。

**2. 关键决策(60s)**

- **为什么选 GSM8K-zh**:答案是数字,reward 可完全规则化(精确匹配),
  不需要训 reward model,在单卡上能跑通完整 RL pipeline
- **为什么选 GRPO 不选 PPO**:PPO 要 4 个模型同时在显存(policy/ref/RM/value),
  12GB 装不下。GRPO 用 group baseline 替代 value model,**省一个模型的显存**,
  数学性能反而更好(DeepSeekMath 2402.03300)
- **为什么 verifiable reward 不偏好对**:偏好对要标注、有 RM bias、训练慢;
  规则化 reward 永远准确、零成本、对推理任务效果远超 DPO

**3. 数字(60s)**

- baseline:Qwen2.5-1.5B-Instruct zero-shot 在 GSM8K-zh test 上 `{baseline_acc}%`
- + GRPO 200 步:`{after_acc}%`,绝对提升 `{delta} pt`
- format 学得快(50 步内 `<think>` 覆盖率 → 95%+),answer 学得慢(150 步后才显著上升)
- 训练耗时 `{train_time}` min,显存峰值 `{vram}` GB

**4. 踩过的坑(60s)**

- 初始 `format_reward=1.0, answer_reward=1.0`:模型学会"刷格式但乱答",reward hacking
  → 改成 `format=0.5, answer=1.5`,format 不再是主要正反馈
- `learning_rate=2e-5` 上来 reward 直接崩溃 → 降到 `5e-6`,稳定后才上升
- `num_generations=8` 显存炸了 → 降到 4 + `max_completion_length=320`,刚好放下
- vLLM 加速采样让训练快 5-10x(`use_vllm=True`),Linux + Qwen2.5 兼容

**5. 后续工作(30s)**

- 换更大 base(Qwen2.5-3B / Qwen3-1.7B),accuracy 会再涨 10+ pt
- 加 reasoning length penalty,避免模型为了 think 而 think
- 复现 DeepSeek-R1 的 cold-start SFT + 多阶段 RL(本节省了第一步)

---

## 11. 延伸 + 论文清单

- 论文:`/paper fetch 2501.12948` (DeepSeek-R1)
- 论文:`/paper fetch 2402.03300` (DeepSeekMath / GRPO)
- 论文:`/paper fetch 2305.18290` (DPO 对照组)
- [Open-R1 完整复现](https://github.com/huggingface/open-r1)
- [Phil Schmid mini-R1 教程](https://www.philschmid.de/mini-deepseek-r1)
- [Karpathy nanochat](https://github.com/karpathy/nanochat) —— 教学风格参考
- 本仓:[`06_alignment/03_grpo_verifiable_reward.ipynb`](../06_alignment/03_grpo_verifiable_reward.ipynb) —— 上游训练细节
- [[Slipbox/deepseek-r1-recipe]] / [[Slipbox/grpo-vs-ppo]] / [[Slipbox/verifiable-reward-design]]
